# Imports


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
from datetime import date as date
import yfinance as yf
import re
from hmmlearn.hmm import GaussianHMM
import matplotlib.pyplot as plt

# Data Management

In [ ]:
# Data extraction
start_date = '2019-09-18'
end_date = date.today()
symbol = 'HBAR-USD'
data = yf.download(symbol, start_date, end_date)
data

In [ ]:
print(data)

In [ ]:
data.head(10)

In [ ]:
data.tail(10)

In [ ]:
data.describe()

In [ ]:
data.info()

In [ ]:
def unify_column_names(name: str) -> str:
    """Converts column names to lowercase, replaces spaces/hyphens with underscores, and handles MultiIndex."""
    return (
        tuple(re.sub(r"[-\s]+", "_", str(n).lower()) for n in name)
        if isinstance(name, tuple)
        else re.sub(r"[-\s]+", "_", str(name).lower())
    )


# Apply transformation to column names
data.columns = data.columns.map(unify_column_names)

In [ ]:
print(f"Unified columns names: \n{data.columns}")

In [ ]:
# Add returns and range
df = data.copy()
df["returns"] = df["close"].pct_change()
df["range"] = (df["high"] / df["low"]) - 1

In [ ]:
df.head(10)

In [ ]:
# Determine the total of NA values and drop them
na_count = df.isna().sum().sum()
if na_count > 0:
    print(f"Dropping {na_count} NA value(s)")
    df.dropna(inplace=True)

In [ ]:
# Structure data
X_train = df[["returns", "range"]]
X_train.head(10)

# HMM Learning

In [ ]:
# Fit model
hmm_model = GaussianHMM(n_components=4, covariance_type="full", n_iter=1000, random_state=42).fit(X_train)
hmm_model_score = hmm_model.score(X_train)
print(f"HMM score: {hmm_model_score}")

In [ ]:
# Check results
hidden_states = hmm_model.predict(X_train)
print(hidden_states[:100])
print(len(hidden_states))

In [ ]:
dir(hmm_model)

In [ ]:
# Regime state means for each feature
hmm_model.means_

In [ ]:
# Regime state covariances
hmm_model.covars_

# Data Visualization

In [ ]:
# Structure prices for plotting

# Initialize lists for each hidden state
labels_0 = []
labels_1 = []
labels_2 = []
labels_3 = []

# Convert close prices to NUmPy array
prices = df["close"].values.astype(float)

# Verify dimensions
print(f"Correct number of rows: {len(prices) == len(hidden_states)}")

In [ ]:
# Assign prices to respective state labels, filling others with NaN
for i, s in enumerate(hidden_states):
    price_value = prices[i].item()

    labels_0.append(price_value if s == 0 else np.nan)
    labels_1.append(price_value if s == 1 else np.nan)
    labels_2.append(price_value if s == 2 else np.nan)
    labels_3.append(price_value if s == 3 else np.nan)

In [ ]:
# Plot chart
fig = plt.figure(figsize=(18, 8))
plt.plot(labels_0, color='green')
plt.plot(labels_1, color='red')
plt.plot(labels_2, color='firebrick')
plt.plot(labels_3, color='goldenrod')
plt.show()